In [0]:
# ============================================================
# PAYCE — Gold Layer Configuration
# Notebook: 03_gold_customer_mart
# ============================================================

CATALOG = "payce"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

print("Gold Confid Loaded Successfully")

In [0]:
spark.table(f"{SILVER}.home_credit").select("NAME_EDUCATION_TYPE").distinct().show(truncate=False)

In [0]:
# ============================================================
# CUSTOMER MART — dim_education
# Distinct education levels from Home Credit
# ============================================================

from pyspark.sql.functions import col, monotonically_increasing_id

dim_education = (
    spark.table(f"{SILVER}.home_credit")
    .select("NAME_EDUCATION_TYPE")
    .distinct()
    .filter(col("NAME_EDUCATION_TYPE").isNotNull())
    .withColumnRenamed("NAME_EDUCATION_TYPE", "education_level")
    .withColumn("education_id", monotonically_increasing_id())
    .select("education_id", "education_level")
)

In [0]:
# Load the table into Gold layer
dim_education.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.dim_education")

In [0]:
# SHOW TABLE
spark.table(f"{GOLD}.dim_education").orderBy("education_id").show()

In [0]:
# ============================================================
# CUSTOMER MART — dim_income_band
# Manually defined income bands (we create these ourselves)
# ============================================================

from pyspark.sql.functions import col

# Define the income bands as a small fixed table
income_bands = [
    (0, "Under 50k", 0, 50000),
    (1, "50k - 100k", 50000, 100000),
    (2, "100k - 150k", 100000, 150000),
    (3, "150k - 200k", 150000, 200000),
    (4, "200k+", 200000, 999999999)
]

dim_income_band = spark.createDataFrame(
    income_bands,
    ["income_band_id", "income_range", "min_income", "max_income"]
)


In [0]:
# Load data into table Gold Layer
dim_income_band.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.dim_income_band")

In [0]:
# SHOW TABLE
spark.table(f"{GOLD}.dim_income_band").orderBy("income_band_id").show()

In [0]:
# ============================================================
# CUSTOMER MART — dim_segment
# Risk tiers we define for customer segmentation
# ============================================================

segments = [
    (0, "Low Risk"),
    (1, "Medium Risk"),
    (2, "High Risk")
]

dim_segment = spark.createDataFrame(
    segments,
    ["segment_id", "segment_name"]
)

In [0]:
# Load this table into Gold
dim_segment.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.dim_segment")

In [0]:
# SHOW TABLE
spark.table(f"{GOLD}.dim_segment").orderBy("segment_id").show()

In [0]:
# ============================================================
# CUSTOMER MART — fct_customer_profile
# Grain: one row per customer
# ============================================================

from pyspark.sql.functions import col, when

# Load sources
hc            = spark.table(f"{SILVER}.home_credit")
dim_education = spark.table(f"{GOLD}.dim_education")
dim_income    = spark.table(f"{GOLD}.dim_income_band")

# Build the customer profile fact
fct_customer = (
    hc.alias("h")
    # join education to get education_id
    .join(dim_education.alias("e"),
          col("h.NAME_EDUCATION_TYPE") == col("e.education_level"), "left")
    # assign income band by range (join where income falls between min and max)
    .join(dim_income.alias("i"),
          (col("h.AMT_INCOME_TOTAL") >= col("i.min_income")) &
          (col("h.AMT_INCOME_TOTAL") <  col("i.max_income")), "left")
    # assign risk segment: defaulted = High; low income non-default = Medium; else Low
    .withColumn(
        "segment_id",
        when(col("h.TARGET") == 1, 2)                                  # defaulted → High Risk
        .when(col("h.AMT_INCOME_TOTAL") < 50000, 1)                    # low income → Medium
        .otherwise(0)                                                  # else → Low Risk
    )
    .select(
        col("h.SK_ID_CURR").alias("customer_id"),
        col("segment_id"),
        col("i.income_band_id"),
        col("e.education_id"),
        col("h.AMT_INCOME_TOTAL").alias("total_income"),
        col("h.age_years"),
        col("h.TARGET").alias("default_flag"),
        col("h.AMT_CREDIT").alias("credit_amount")
    )
)

In [0]:
# Write to Gold
fct_customer.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD}.fct_customer_profile")

In [0]:
# Confirm
print(f"Rows: {spark.table(f'{GOLD}.fct_customer_profile').count():,}")
spark.table(f"{GOLD}.fct_customer_profile").show(5)
